# GRU kapasitesi: hidden128 → hidden64

Yeni A100 oturumu; icinbedava34@gmail.com hesabı.
Yalnız GRU gizli boyutu değişir. Katman2, fusion128, dropout0.2, LR0.001 ve altı özellik aynı.
Önceki CNN-GRU çalışmasının114 GRU kontrolü salt okunur doğrulanır; yeniden eğitilmez.
Hidden64 için38 fold ×3 seed =114 yeni eğitim.
Train5040; validation528 seçim +24 boşluk +528 değerlendirme,465 hedef/bölüm.
Epoch seçim BCE ile; eşikler0.5 ve seçim üst%5/%10. Final test yok.


In [ ]:
import os
os.environ['MPLBACKEND']='Agg'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
LEGACY=Path('/content/drive/MyDrive/yeniBot/advisor_experiments/colab_v1')
SOURCE=Path('/content/drive/MyDrive/yeniBot/advisor_experiments/feature_comparison_v1/a100_001')
CONTROL=Path('/content/drive/MyDrive/yeniBot/advisor_experiments/cnn_gru_v1/a6_a100_001')
ROOT=Path('/content/drive/MyDrive/yeniBot/advisor_experiments/gru_capacity_v1/hidden64_a100_001')
AUTO_DISCONNECT=True
RELEASE='advisor-gru-capacity-v1.0.0'
REPO=Path('/content/advisor_feature_comparison_repo')
RUNTIME=Path('/content/advisor_feature_comparison_runtime')
SUCCESS=set()


## İzole ortam
Eski çalışmayla aynı kilitli Python/paketler ve A100 kullanılır. Eski veri salt okunur; yeniden veri çekilmez.


In [ ]:
import subprocess,sys,json,hashlib
from datetime import datetime,timezone
SESSION=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
LOGS=ROOT/'sessions'/SESSION
LOGS.mkdir(parents=True,exist_ok=True)
ENV=dict(os.environ)
for key in ['PYTHONPATH','PYTHONHOME','PYTHONSTARTUP','IPYTHONDIR','JUPYTER_CONFIG_DIR']:
    ENV.pop(key,None)
ENV.update(MPLBACKEND='Agg',PYTHONNOUSERSITE='1',CUBLAS_WORKSPACE_CONFIG=':4096:8',OMP_NUM_THREADS='4',MKL_NUM_THREADS='4')
def logged(command,name):
    with (LOGS/name).open('a',encoding='utf-8') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=ENV)
        try:
            for line in process.stdout:
                print(line,end='');log.write(line);log.flush()
            code=process.wait()
            if code:raise subprocess.CalledProcessError(code,command)
        except BaseException:
            if process.poll() is None:process.terminate();process.wait()
            raise
URL='https://github.com/umutergul74/YeniBotDeney.git'
if not REPO.exists():
    logged(['git','clone','--branch',RELEASE,'--depth','1',URL,str(REPO)],'setup.log')
def git(*args):return subprocess.check_output(['git','-C',str(REPO),*args],text=True).strip()
assert git('remote','get-url','origin')==URL
assert git('rev-parse','HEAD')==git('rev-parse',RELEASE+'^{commit}')
assert not git('status','--porcelain','--untracked-files=no')
receipt=RUNTIME/'environment_receipt.json'
logged([sys.executable,'-I',str(REPO/'scripts/bootstrap_advisor.py'),'--repo',str(REPO),
    '--runtime',str(RUNTIME),'--cache',str(ROOT/'bootstrap_cache'),'--receipt',str(receipt)],'setup.log')
PYTHON=json.loads(receipt.read_text())['python']
def step(action):
    order=['prepare','train','report']
    SUCCESS.difference_update(order[order.index(action):])
    command=[PYTHON,'-I',str(REPO/'scripts/advisor_capacity.py'),action,
        '--root',str(ROOT),'--source',str(SOURCE),'--control',str(CONTROL)]
    logged(command,action+'.log')
    SUCCESS.add(action)


##114 eski kontrolün dosyalarını, ortamını ve bölmelerini doğrula


In [ ]:
step('prepare')


## Yalnız hidden64 için114 eğitim
Kontrol yeniden eğitilmez. Kesinti sonrası aynı workspace ile devam eder.


In [ ]:
step('train')


## Seçimden türetilen eşiklerle değerlendirme
0.5 referansı ve seçim bölümünün üst %5/%10 skor eşikleri. Otomatik kazanan yok.


In [ ]:
step('report')
from IPython.display import display, Image
display(Image(filename=str(ROOT/'reports/comparison.png')))
print((ROOT/'reports/summary.csv').read_text())
print('İnceleme ZIP:',ROOT/'reports/capacity_review.zip')


## Drive doğrulaması ve otomatik kapanış
Dosyalar yeniden bağlanınca doğrulanmadan oturum kapanmaz.


In [ ]:
if 'report' not in SUCCESS:raise RuntimeError('Önce rapor hücresini tamamlayın')
def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
expected={str(p.relative_to(ROOT)):digest(p) for p in ROOT.rglob('*') if p.is_file()}
if not expected:raise RuntimeError('Workspace boş')
drive.flush_and_unmount()
drive.mount('/content/drive')
for relative,checksum in expected.items():
    p=ROOT/relative
    if not p.is_file() or digest(p)!=checksum:raise RuntimeError('Drive doğrulaması başarısız: '+relative)
print('Drive dosyaları yeniden bağlandıktan sonra doğrulandı. Raporlar:',ROOT/'reports')
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()
